# Linting et refactoring

PEP 8 fixe le style du code Python : indentation, espaces, longueur de ligne et nommage.
flake8 signale les écarts par analyse statique, sans exécuter le programme.
black reformate le code lorsque la correction ne change pas le comportement.
Ce notebook illustre la norme, puis les deux outils, avec une longueur de ligne de 120 caractères.
La référence est [PEP 8](https://www.python.org/dev/peps/pep-0008/). Les docstrings suivent [PEP 257](https://www.python.org/dev/peps/pep-0257/).

## Norme PEP 8

PEP 8 (*Style Guide for Python Code*, juillet 2001) est le guide de style adopté pour le code Python.
Un projet peut assouplir certaines règles, notamment la longueur de ligne, fixée à 79 caractères dans la norme.

Après un saut de ligne, les arguments s'alignent sur le premier argument.
L'indentation est de 4 espaces par niveau.
Un fichier qui mélange tabulations et espaces ne s'exécute pas.

Autres règles d'espacement : un espace après une virgule, aucun avant ; aucun espace à l'intérieur des crochets ; un espace autour des opérateurs binaires ; aucun espace contre les parenthèses.

In [ ]:
def fonction(a, b, c, d):
    return a * b * c * d

# Pas bien car les arguments ne sont pas alignés en colonne
resultat = fonction(1, 2,
                   3, 4)
# Bien
resultat = fonction(1, 2,
                    3, 4)

l = range(10)

# Pas bien
var1,var2 = l[1 ],{ 'var' : 4 }

# Bien
var1, var2 = l[1], {'var': 4}

## Analyse avec flake8

flake8 s'appuie sur PEP 8 et peut appliquer des règles supplémentaires.
Les codes `E2xx` concernent les espaces. `E741` signale un nom ambigu, ici `l`.
Un nom utilisé alors qu'il n'est pas défini est un code `F821` : l'exécution échouerait.

Le premier fichier contient les écarts. Le second est la version corrigée.
La liste des codes est publiée par [pycodestyle](https://pycodestyle.pycqa.org/en/latest/intro.html#error-codes).

`--ignore` retire un code de l'analyse. `E401` correspond à plusieurs imports sur la même ligne.
Les autres codes restent signalés, par exemple un import inutilisé.

In [ ]:
%%writefile /tmp/flake8_1.py
l= range(10 )
var1,var2 = l[1 ],{ 'var' : 4 }

In [ ]:
!flake8 /tmp/flake8_1.py

In [ ]:
%%writefile /tmp/flake8_1.py
liste = range(10)
var1, var2 = liste[1], {'var': 4}

In [ ]:
!flake8 /tmp/flake8_1.py

In [ ]:
%%writefile /tmp/flake8_2.py
import os, sys

filename = open(os.path.join("/home", "/fichier.txt"))

In [ ]:
!flake8 --ignore E401 /tmp/flake8_2.py

## Reformatage avec black

black corrige indentation, espaces et retours à la ligne.
Il ne renomme pas les variables et ne modifie pas les imports. Ces points restent à flake8.
L'usage est d'enchaîner black, puis flake8.

La longueur par défaut est 79 caractères.
L'option `-l 120` aligne black sur la limite retenue pour ce projet. Le code `E501` (ligne trop longue au sens strict de PEP 8) est alors hors sujet si flake8 utilise la même limite.

In [ ]:
%%writefile /tmp/black_2.py
from lightgbm.sklearn import LGBMClassifier
from hyperopt import hp, tpe, fmin
MODEL_SPECS = {
    "name": "LightGBM",
    "class": LGBMClassifier,
    "max_evals": 20,
    "params": {
        "learning_rate": hp.uniform("learning_rate", 0.001, 1),
        "num_iterations": hp.quniform("num_iterations", 100, 1000, 20),
        "max_depth": hp.quniform("max_depth", 4, 12, 6),
        "num_leaves": hp.quniform("num_leaves", 8, 128, 10),
        "colsample_bytree": hp.uniform("colsample_bytree", 0.3, 1),
        "subsample": hp.uniform("subsample", 0.5, 1),
        "min_child_samples": hp.quniform("min_child_samples", 1, 20, 10),
        "reg_alpha": hp.choice("reg_alpha", [0, 1e-1, 1, 2, 5, 10]),
        "reg_lambda": hp.choice("reg_lambda", [0, 1e-1, 1, 2, 5, 10]),
    },
    "override_schemas": {
        "num_leaves": int, "min_child_samples": int, "max_depth": int, "num_iterations": int
    }
}

In [ ]:
# On augmente la taille des lignes à 120
!black -l 120 /tmp/black_2.py
print(open("/tmp/black_2.py", "r").read())